In [1]:
import json
import os
import tqdm
import pandas as pd
from IPython.display import display as ipython_display

pd.options.display.max_columns = 500

In [2]:
ANSWER_KEY = "answer-qwen3-32b-think"

In [13]:
def compute_score(path, k=10):
    res = {}
    with open(path) as f:
        for line in f:
            x = json.loads(line)
            lang = x["lang"]
            if lang not in res:
                res[lang] = {"mrr": 0, "n": 0}
            top = sorted(x["candidates"], key=lambda d: -d["score"])[:k]
            for i, d in enumerate(top):
                if answer_to_label_think(d[ANSWER_KEY]) == 1:
                    res[lang]["mrr"] += 1 / (1 + i)
                    break
            res[lang]["n"] += 1
    return {k: v["mrr"] / v["n"] for k, v in res.items()}

def answer_to_label(a):
    a = a.lower().strip()
    if a.startswith("yes"):
        return 1
    elif a.startswith("no"):
        return 0
    else:
        return -1

def answer_to_label_think(a):
    tag = "</think>"
    if tag in a:
        i = a.index(tag)
        return answer_to_label(a[i+len(tag):])
    else:
        return -1

def display(df):
    """
    version, bench, lang, score
    """
    print("by (bench, lang) pair:")
    tmp = df.pivot(index="version", columns=["bench", "lang"], values="score").sort_index(axis=1, level=[0, 1])
    tmp["AVG"] = tmp.mean(1)
    ipython_display(tmp.style.highlight_max(axis=0))
    print()
    print("by bench, avg by lang:")
    tmp = df \
        .groupby(["version", "bench"])["score"] \
        .mean() \
        .reset_index() \
        .pivot(index="version", columns="bench", values="score") \
        .sort_index(axis=1)
    tmp["AVG"] = tmp.mean(1)
    ipython_display(tmp.style.highlight_max(axis=0))

In [ ]:
preds_dir = "/path/to/preds/dir"
# preds_dir
#   qwen--qwen3-emb-06b
#     cosqa.jsonl
#     ...
#  ...

versions = [
    "qwen--qwen3-emb-06b",
    "qwen--qwen3-emb-4b",
    "qwen--qwen3-emb-8b",
    "codesage--codesage-large-v2",
    "jinaai--jina-embeddings-v2-base-code",
    "infly--inf-retriever-v1-1.5b",
    "infly--inf-retriever-v1",
    "jinaai--jina-embeddings-v4-vllm-code",
    "nomic-ai--CodeRankEmbed",
    "tf-idf--default-analyzer",
    "tf-idf--custom-analyzer",
    "megacode-emb-v1-0.5b-pt",
    "megacode-emb-v1-1.5b-pt",
    "megacode-emb-v1-3b-pt",
    "megacode-emb-v1-7b-pt",
    "megacode-emb-v1-0.5b",
    "megacode-emb-v1-1.5b",
    "megacode-emb-v1-3b",
    "megacode-emb-v1-7b"
]
benches = [
    "cosqa", 
    "cosqa-plus",
    "repoqa-with-parsed-query", 
    "csn-adv", 
    "csn-orig", 
    "csn-query",
]

In [6]:
# check that all preds were annotated
for v in versions:
    for b in benches:
        path = os.path.join(preds_dir, v, b + ".jsonl")
        if not os.path.exists(path):
            print(f"[{v}; {b}] no file")
            continue
        with open(path) as f:
            x = json.loads(next(f))
            d = x["candidates"][0]
            if ANSWER_KEY not in d:
                print(f"[{v}; {b}] no annotation")

In [16]:
df = []
for v in tqdm.tqdm(versions):
    for b in benches:
        lang2score = compute_score(os.path.join(preds_dir, v, b + ".jsonl"), k=10)
        for lang, score in lang2score.items():
            df.append({"version": v, "bench": b, "lang": lang, "score": score})
df = pd.DataFrame(df)

100%|██████████| 19/19 [02:49<00:00,  8.92s/it]


In [17]:
# overall
display(df)

by (bench, lang) pair:



by bench, avg by lang:


bench,cosqa,cosqa-plus,csn-adv,csn-orig,csn-query,repoqa-with-parsed-query,AVG
version,,,,,,,
codesage--codesage-large-v2,0.747481,0.793689,0.860622,0.880837,0.773734,0.935433,0.831966
megacode-emb-v1-0.5b,0.827603,0.881778,0.899819,0.905563,0.825911,0.974044,0.885786
megacode-emb-v1-0.5b-pt,0.799779,0.825533,0.848169,0.899344,0.797153,0.966534,0.856085
megacode-emb-v1-1.5b,0.848948,0.886933,0.930876,0.913630,0.825890,0.982431,0.898118
megacode-emb-v1-1.5b-pt,0.823156,0.866000,0.901199,0.908219,0.788529,0.972837,0.876657
megacode-emb-v1-3b,0.858002,0.896743,0.934397,0.914845,0.837887,0.980694,0.903761
megacode-emb-v1-3b-pt,0.833848,0.862089,0.906457,0.910975,0.811832,0.973905,0.883184
megacode-emb-v1-7b,0.861755,0.904610,0.943837,0.919272,0.832253,0.985667,0.907899
megacode-emb-v1-7b-pt,0.829746,0.865029,0.922307,0.913762,0.807256,0.975069,0.885528


In [18]:
# lexical
subset = [
    "tf-idf--default-analyzer",
    "tf-idf--custom-analyzer"
]
display(df[df["version"].isin(subset)])

by (bench, lang) pair:



by bench, avg by lang:


bench,cosqa,cosqa-plus,csn-adv,csn-orig,csn-query,repoqa-with-parsed-query,AVG
version,,,,,,,
tf-idf--custom-analyzer,0.453846,0.554829,0.440426,0.664059,0.628023,0.530366,0.545258
tf-idf--default-analyzer,0.424353,0.461879,0.317107,0.503783,0.560503,0.402257,0.444981


In [ ]:
# 0.5b
subset = [
    "nomic-ai--CodeRankEmbed",
    "jinaai--jina-embeddings-v2-base-code",
    "qwen--qwen3-emb-06b",
    "megacode-emb-v1-0.5b-pt",
    "megacode-emb-v1-0.5b",
]
display(df[df["version"].isin(subset)])

by (bench, lang) pair:



by bench, avg by lang:


bench,cosqa,cosqa-plus,csn-adv,csn-orig,csn-query,repoqa-with-parsed-query,AVG
version,,,,,,,
megacode-emb-v1-0.5b,0.827603,0.881778,0.899819,0.905563,0.825911,0.974044,0.885786
megacode-emb-v1-0.5b-pt,0.799779,0.825533,0.848169,0.899344,0.797153,0.966534,0.856085
jinaai--jina-embeddings-v2-base-code,0.768512,0.865210,0.768456,0.879919,0.792089,0.910780,0.830827
nomic-ai--CodeRankEmbed,0.748601,0.827067,0.824220,0.880431,0.783261,0.937328,0.833485
qwen--qwen3-emb-06b,0.801574,0.852133,0.838600,0.895716,0.781948,0.945319,0.852548


In [ ]:
# 1.5b
subset = [
    "codesage--codesage-large-v2",
    "infly--inf-retriever-v1",
    "megacode-emb-v1-1.5b-pt",
    "megacode-emb-v1-1.5b",
]
display(df[df["version"].isin(subset)])

by (bench, lang) pair:



by bench, avg by lang:


bench,cosqa,cosqa-plus,csn-adv,csn-orig,csn-query,repoqa-with-parsed-query,AVG
version,,,,,,,
codesage--codesage-large-v2,0.747481,0.793689,0.860622,0.880837,0.773734,0.935433,0.831966
megacode-emb-v1-1.5b,0.848948,0.886933,0.930876,0.913630,0.825890,0.982431,0.898118
megacode-emb-v1-1.5b-pt,0.823156,0.866000,0.901199,0.908219,0.788529,0.972837,0.876657
infly--inf-retriever-v1,0.783729,0.832556,0.817910,0.888250,0.776458,0.947097,0.841000


In [ ]:
# 3b
subset = [
    "jinaai--jina-embeddings-v4-vllm-code",
    "qwen--qwen3-emb-4b",
    "megacode-emb-v1-3b-pt",
    "megacode-emb-v1-3b",
]
display(df[df["version"].isin(subset)])

by (bench, lang) pair:



by bench, avg by lang:


bench,cosqa,cosqa-plus,csn-adv,csn-orig,csn-query,repoqa-with-parsed-query,AVG
version,,,,,,,
megacode-emb-v1-3b,0.858002,0.896743,0.934397,0.914845,0.837887,0.980694,0.903761
megacode-emb-v1-3b-pt,0.833848,0.862089,0.906457,0.910975,0.811832,0.973905,0.883184
jinaai--jina-embeddings-v4-vllm-code,0.740593,0.807467,0.775959,0.799518,0.688323,0.871476,0.780556
qwen--qwen3-emb-4b,0.822067,0.825000,0.885562,0.905338,0.778594,0.969546,0.864351


In [ ]:
# 7b
subset = [
    "infly--inf-retriever-v1",
    "qwen--qwen3-emb-8b",
    "megacode-emb-v1-7b-pt",
    "megacode-emb-v1-7b"
]
display(df[df["version"].isin(subset)])

by (bench, lang) pair:



by bench, avg by lang:


bench,cosqa,cosqa-plus,csn-adv,csn-orig,csn-query,repoqa-with-parsed-query,AVG
version,,,,,,,
megacode-emb-v1-7b,0.861755,0.904610,0.943837,0.919272,0.832253,0.985667,0.907899
megacode-emb-v1-7b-pt,0.829746,0.865029,0.922307,0.913762,0.807256,0.975069,0.885528
infly--inf-retriever-v1,0.783729,0.832556,0.817910,0.888250,0.776458,0.947097,0.841000
qwen--qwen3-emb-8b,0.819548,0.804756,0.892422,0.906925,0.786402,0.969569,0.863270
